# Step 3: Deep Learning — Spectrograms & CNN (5-Class Emotion Classification)

## Objective
We feed the **actual visual pattern of the sound wave** into a neural network. 
We convert the audio waveform into a **Mel-Spectrogram** (a 2D image of frequency and time) and train a **Convolutional Neural Network (CNN)**.

### The 5-Class Approach
Instead of grouping into broad sentiments, we want to classify specific emotions. 
To avoid dropping the largest portion of our dataset, we include **Frustrated** as its own distinct class alongside the standard four:
1. **Angry**
2. **Happy** (Merged with Excited)
3. **Sad**
4. **Neutral**
5. **Frustrated**

This makes it a 5-class classification problem.


## 0. Install Dependencies & Imports

In [ ]:
# !pip install torch torchaudio torchvision scikit-learn


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T

from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print(f"Using device: {device}")


## 1. Dataset Preparation & 5-Class Mapping

All inputs must be the same size, so we define a fixed audio length of **4 seconds**.


In [ ]:
print("Loading IEMOCAP dataset...")
ds = load_dataset("AbstractTTS/IEMOCAP", split="train")

# 5-Class Emotion Mapping
LABEL_MAP = {
    "angry": 0,
    "happy": 1,
    "excited": 1,      # Merged with happy
    "sad": 2,
    "neutral": 3,
    "frustrated": 4
}
CLASS_NAMES = ["Angry", "Happy", "Sad", "Neutral", "Frustrated"]

# Parameters for Audio Processing
SAMPLE_RATE = 16000
MAX_SECONDS = 4
MAX_LENGTH = SAMPLE_RATE * MAX_SECONDS

class IEMOCAPDataset(Dataset):
    def __init__(self, hf_dataset, target_sessions):
        self.data = []
        
        print(f"Preparing dataset for sessions: {target_sessions}...")
        for item in tqdm(hf_dataset):
            raw_emo = item.get("major_emotion", "")
            if raw_emo not in LABEL_MAP:
                continue
                
            fname = item.get("file", "")
            session_id = fname.split("/")[0] if "/" in fname else fname[:5]
            
            # LOSO Split
            if session_id not in target_sessions:
                continue
                
            y = np.array(item["audio"]["array"], dtype=np.float32)
            
            # Padding / Truncating to 4 seconds
            if len(y) > MAX_LENGTH:
                y = y[:MAX_LENGTH]
            else:
                pad_len = MAX_LENGTH - len(y)
                y = np.pad(y, (0, pad_len), "constant")
                
            self.data.append({
                "audio": torch.tensor(y),
                "label": LABEL_MAP[raw_emo]
            })
            
        self.melspec_transform = T.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_mels=64,
            n_fft=1024,
            hop_length=512
        )
        self.amplitude_to_db = T.AmplitudeToDB()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        mel = self.melspec_transform(item["audio"])
        mel_db = self.amplitude_to_db(mel)
        mel_db = mel_db.unsqueeze(0) # Channel dimension
        return mel_db, item["label"]

# Train: Sessions 1, 2, 3, 4 | Test: Session 5
train_dataset = IEMOCAPDataset(ds, target_sessions=["Ses01", "Ses02", "Ses03", "Ses04"])
test_dataset  = IEMOCAPDataset(ds, target_sessions=["Ses05"])

print(f"\nTrain size: {len(train_dataset)} | Test size: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


## 2. Visualising the Model's Input

In [ ]:
sample_batch, sample_labels = next(iter(train_loader))
sample_img = sample_batch[0]
sample_lbl = sample_labels[0].item()

plt.figure(figsize=(10, 4))
plt.imshow(sample_img.squeeze().numpy(), cmap='magma', origin='lower', aspect='auto')
plt.colorbar(format='%+2.0f dB')
plt.title(f"CNN Input (Mel-Spectrogram) | Label: {CLASS_NAMES[sample_lbl]}", fontweight="bold")
plt.ylabel("Mel Frequency Bins"); plt.xlabel("Time Frames")
plt.tight_layout()
plt.show()


## 3. CNN Architecture

In [ ]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes=5): # Now 5 classes!
        super(EmotionCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 16, 3, 1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(2)
        
        self.conv2 = nn.Conv2d(16, 32, 3, 1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(2)
        
        self.conv3 = nn.Conv2d(32, 64, 3, 1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(2)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
        self.fc1 = nn.Linear(64 * 6 * 14, 128) 
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu(self.bn3(self.conv3(x))))
        
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = EmotionCNN(num_classes=5).to(device)
print(model)


## 4. Training Loop (10 Epochs)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 10

print("Starting training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {100 * correct / total:.2f}%")
print("Training finished! 🚀")


## 5. Evaluation & Confusion Matrix

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Test Accuracy: {accuracy_score(all_labels, all_preds)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix (5-Class Emotion CNN)", fontweight="bold")
plt.ylabel('True Emotion'); plt.xlabel('Predicted Emotion')
plt.show()
